In [ ]:
# Common imports + DES (Modelica) translator pieces.

import pandas as pd
from geojson_modelica_translator.modelica.GMT_Lib.DHC.DHC_5G_WH_GHX_HPTrio_VariableDist import (
    DHC5GWasteHeatGHXwithHPTrioVariableDist,
)
from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner
from geojson_modelica_translator.system_parameters.system_parameters import (
    SystemParameters,
)
import time

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
    copy_activity_to_new_location,
    create_subset_geojson,
    create_subset_scenario_csv,
    summarize_openmodelica_timing,
)

import re

from buildingspy.io.outputfile import Reader

from lib.helpers import plot_cascade_temperature_challenge


# -- Execution mode -----------------------------------------------------------
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# Change this to keep models/results outside the notebook folder (recommended).
# Examples: "../esbe_2026" (one level up) or "esbe_2026" (inside this folder).
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

### Activity 07b: TEN [Loop Order] n-Building Scenario
* Before we create a full annual TEN, we need to explore the impacts of building order on the 5G scenario. This notebook will create a 5G scenario for a subselection of buildings.
* When the number of buildings increases, the number of differential algebraic equations increases, thus significantly increasing simulation time. For this workflow, do not analyze more than 4 buildings; runtime will be about 20 minutes per scenario.
* Using the class project building, and looking at annual net loads as well as the [Zarin-Pass metric](https://escholarship.org/content/qt4895898c/qt4895898c.pdf)
  * Tested every selected-building combination at each hourly timestep using `H_t = heating_t + DHW_t` and `C_t = abs(cooling_t)`, selected the building set because its annual net offset `sum(H_t - C_t)` was closest to zero (+15.6 MWh), and checked simultaneous overlap with the Zarin Pass metric `sum(min(H_t, C_t)) / sum(max(H_t, C_t)) = 0.189`
  * Best order for the selected buildings: res_6 -> res_5 -> office_7 -> school_1 -> hotel_1
  * Worst would be the reverse order: hotel_1 -> school_1 -> office_7 -> res_5 -> res_6


    | Building  | Net Energy (MWh) | Dominance         |
    |-----------|------------------:|-------------------|
    | hotel_1   |        -553,447   | cooling-dominant  |
    | school_1  |        +462,417   | heating-dominant  |
    | office_7  |         +76,382   | heating-dominant  |
    | res_5     |          +8,158   | heating-dominant  |
    | res_6     |          +6,505   | heating-dominant  |

In [ ]:
# Copy activity_03/coincident to activity_07a/coincident (deep copy of the full folder tree).
source_dir = paths.activity_dir("activity_03") / "coincident"
activity_4_dir = paths.activity_dir("activity_07a")
dest_dir = activity_4_dir / "coincident"
copy_activity_to_new_location(source_dir, dest_dir)

# Set up the coincident project from activity_07a/coincident
uo_coincident = UO(activity_4_dir, "coincident", template_dir=template_data_dir)

# Weather data is copied over from the activity_03/coincident project. No
# need to update the weather information.

# Common paths used by the DES (Modelica) steps below.
# Saved as relative-style paths because they're consumed from within Docker.
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# For Python-based DES creation with the TEN template, there isn't a need to create
# a _loop_order file.

### 5G: Run with individual loads (Only run for a couple days, takes a long time to run)

* The default loop order is building 1 -> 23. 
* Order is in the `Districts.district` array in the generated model.
* Changing order can change the downstream node temperatures and impact individual building energy consumption.

In [ ]:
# Define the n-building subset and the two loop-order scenarios to compare.
#
# In a 5G district loop the first building has first access to the network
# resource. Placing heating-dominant buildings late in the loop lets them
# benefit from waste heat shed by the upstream cooling-dominant hotel.
#
# To experiment with a different subset, change SUBSET_BUILDINGS.
# To try a different order, change the list inside LOOP_ORDER_SCENARIOS.
#
# Important: do not analyze more than 4 buildings in this workflow.
# Runtime is about 20 minutes per scenario, so bigger runs scale quickly.

LOOP_ORDER_SCENARIOS = {
    # Best: heating-dominant buildings are served first; hotel anchors the end.
    "five_g_subset_best_loop_order": [
        # "res_6",
        "res_5",
        "office_7",
        "school_1",
        "hotel_1",
    ],
    # Worst: cooling-dominant building serves first, starving heating-dominant buildings.
    "five_g_subset_worst_loop_order": [
        "hotel_1",
        "school_1",
        "office_7",
        "res_5",
        # "res_6",
    ],
}

# Run only these scenarios during iteration.
ACTIVE_SCENARIOS = [
    "five_g_subset_worst_loop_order",
    "five_g_subset_best_loop_order",
]

# Time windows to simulate (label, start_seconds, stop_seconds).
# All times are seconds from Jan 1 00:00 in a non-leap year.
SIM_RUN_PERIODS = [
    ("march_1_2", 5_097_600, 5_270_400),  # Mar 1 00:00 → Mar 3 00:00  (2 days)
    ("june_20_21", 14_688_000, 14_860_800),  # Jun 20 00:00 → Jun 22 00:00 (2 days)
    ("dec_10_11", 29_635_200, 29_808_000),  # Dec 10 00:00 → Dec 12 00:00 (2 days)
]

print("Loop order scenarios:")
for name, order in LOOP_ORDER_SCENARIOS.items():
    print(f" {name}: {' -> '.join(order)}")
print(f"\nActive scenarios: {ACTIVE_SCENARIOS}")
print(f"Time windows: {[w[0] for w in SIM_RUN_PERIODS]}")

In [ ]:
# IMPORTANT: Modelica uses directory/scenario name in the object path, so we must
# rebuild the model under a specific scenario name. The run-period label
# also goes into the directory name so each timed run stays unique.
#
# Set RUN_MODEL = False to do a dry run that only creates and checks the
# directory structure without launching simulations. In that mode, the
# mr.run_in_docker(...) call stays commented out.

# Set RUN_MODEL to False to skip running models which can be used
# to verify that the models were generated correctly, or to run in OMEdit
RUN_MODEL = True

mr = ModelicaRunner()

for label, start_time, stop_time in SIM_RUN_PERIODS:
    print(f"compile_and_run window: {label} ({start_time} -> {stop_time})")

    for des_scenario_name in ACTIVE_SCENARIOS:
        loop_order = LOOP_ORDER_SCENARIOS[des_scenario_name]
        run_name = f"{des_scenario_name}_{label}"

        # 1) Create specific subset inputs.
        subset_geojson = activity_4_dir / f"{run_name}_features.json"
        create_subset_geojson(feature_path, loop_order, subset_geojson)

        subset_csv = activity_4_dir / f"{run_name}_scenario.csv"
        create_subset_scenario_csv(scenario_path, loop_order, subset_csv)

        # 2) Create specific system parameters.
        uo_analysis_baseline_results = (
            uo_coincident.project_path / "run" / "baseline_scenario"
        )
        sys_param_file = activity_4_dir / f"{run_name}_system_params_5g.json"
        run_dir = activity_4_dir / run_name
        if run_dir.exists():
            # print a warning instead of deleting the directory to avoid losing previous results.
            print(f"Warning: directory {run_dir} already exists. Skipping deletion.")
            print("  If you want to rebuild the model, manually delete the directory")
            continue

        system_parameter = SystemParameters()
        system_parameter.csv_to_sys_param(
            model_type="time_series",
            sys_param_filename=sys_param_file,
            scenario_dir=uo_analysis_baseline_results,
            feature_file=subset_geojson,
            district_type="5G",
            relative_path=None,
            modelica_load_filename="modelica.mos",
        )

        # 3) Build model with a matching object path name.
        model = DHC5GWasteHeatGHXwithHPTrioVariableDist(system_parameter)
        model.build_from_template(activity_4_dir, run_name)

        model_name = f"{run_name}.Districts.district"
        print(f"\n--- compile_and_run benchmark: {run_name} ---")

        # 4) Run compile_and_run and capture timing.
        t0 = time.perf_counter()
        # When RUN_MODEL is False, keep the run call commented out.
        if RUN_MODEL:
            success, results_path = mr.run_in_docker(
                "compile_and_run",
                model_name,
                file_to_load=run_dir / "package.mo",
                run_path=run_dir,
                step_size=900,
                start_time=start_time,
                stop_time=stop_time,
            )
        else:
            success = True
            results_path = run_dir / "run" / run_name

        wall_s = time.perf_counter() - t0

        summarize_openmodelica_timing(success, wall_s, results_path)

In [ ]:
# Export loop-temperature plots for every created scenario MAT into activity_07a/_results_summary.
# Uses the helper in lib/helpers.py: plot_cascade_temperature_challenge(...)
summary_dir = activity_4_dir / "_results_summary"
summary_dir.mkdir(parents=True, exist_ok=True)

# Discover all district MAT files generated for active scenarios.
mat_files = sorted(activity_4_dir.glob("**/*.Districts.district_res.mat"))
mat_files = [
    p
    for p in mat_files
    if any(scenario_name in p.as_posix() for scenario_name in ACTIVE_SCENARIOS)
]

print(f"Found {len(mat_files)} MAT file(s) to process")

SUP_RE = re.compile(r"^dis\.con\[(\d+)\]\.junConSup\.vol\.T$")
RET_RE = re.compile(r"^dis\.con\[(\d+)\]\.junConRet\.vol\.T$")

exported = []

for mat_path in mat_files:
    reader = Reader(str(mat_path), "dymola")
    var_names = reader.varNames()

    # Find all building ids that have both supply and return temperatures.
    sup_ids = {int(m.group(1)) for v in var_names if (m := SUP_RE.match(v))}
    ret_ids = {int(m.group(1)) for v in var_names if (m := RET_RE.match(v))}
    building_ids = sorted(sup_ids & ret_ids)

    if not building_ids:
        print(
            f"Skipping {mat_path.name}: no dis.con[*] supply/return temperatures found"
        )
        continue

    first_supply = f"dis.con[{building_ids[0]}].junConSup.vol.T"
    time_s, _ = reader.values(first_supply)

    data = {
        "time_s": time_s,
        "hours_from_start": (time_s - time_s[0]) / 3600.0,
    }

    # Convert K to C for plotting.
    for bid in building_ids:
        _, sup_vals = reader.values(f"dis.con[{bid}].junConSup.vol.T")
        _, ret_vals = reader.values(f"dis.con[{bid}].junConRet.vol.T")
        data[f"Building {bid} supply"] = sup_vals - 273.15
        data[f"Building {bid} return"] = ret_vals - 273.15

    df = pd.DataFrame(data)

    # Match earlier convention: ignore first hour warm-up.
    df = df[df["hours_from_start"] >= 1.0].copy()

    run_name = mat_path.stem.replace(".Districts.district_res", "")
    csv_out = summary_dir / f"{run_name}_cascade_temperature_metrics.csv"
    png_out = summary_dir / f"{run_name}_loop_temperature_challenge.png"

    df.to_csv(csv_out, index=False)
    plot_cascade_temperature_challenge(csv_out, output_path=png_out, show=False)

    exported.append((run_name, csv_out, png_out))

print("\nExported loop-temperature artifacts:")
for run_name, csv_out, png_out in exported:
    print(f"  {run_name}")
    print(f"    csv: {csv_out}")
    print(f"    png: {png_out}")

In [ ]:
# Compare best vs worst loop temperatures by date and export side-by-side plots.
from pathlib import Path
import matplotlib.pyplot as plt

best_scenario = "five_g_subset_best_loop_order"
worst_scenario = "five_g_subset_worst_loop_order"

compare_dir = summary_dir / "best_vs_worst_by_date"
compare_dir.mkdir(parents=True, exist_ok=True)

SUP_RE = re.compile(r"^dis\.con\[(\d+)\]\.junConSup\.vol\.T$")
RET_RE = re.compile(r"^dis\.con\[(\d+)\]\.junConRet\.vol\.T$")


def build_metrics_csv_from_mat(mat_path: Path, csv_out: Path) -> Path:
    reader = Reader(str(mat_path), "dymola")
    var_names = reader.varNames()

    sup_ids = {int(m.group(1)) for v in var_names if (m := SUP_RE.match(v))}
    ret_ids = {int(m.group(1)) for v in var_names if (m := RET_RE.match(v))}
    building_ids = sorted(sup_ids & ret_ids)
    if not building_ids:
        raise ValueError(f"No loop supply/return vars found in {mat_path}")

    first_supply = f"dis.con[{building_ids[0]}].junConSup.vol.T"
    time_s, _ = reader.values(first_supply)

    data = {
        "time_s": time_s,
        "hours_from_start": (time_s - time_s[0]) / 3600.0,
    }

    for bid in building_ids:
        _, sup_vals = reader.values(f"dis.con[{bid}].junConSup.vol.T")
        _, ret_vals = reader.values(f"dis.con[{bid}].junConRet.vol.T")
        data[f"Building {bid} supply"] = sup_vals - 273.15
        data[f"Building {bid} return"] = ret_vals - 273.15

    df = pd.DataFrame(data)
    df = df[df["hours_from_start"] >= 1.0].copy()

    csv_out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_out, index=False)
    return csv_out


def ensure_metrics_csv(run_name: str) -> Path:
    csv_out = summary_dir / f"{run_name}_cascade_temperature_metrics.csv"
    if csv_out.exists():
        return csv_out

    mat_path = (
        activity_4_dir
        / run_name
        / f"{run_name}.Districts.district_results"
        / f"{run_name}.Districts.district_res.mat"
    )
    if not mat_path.exists():
        raise FileNotFoundError(f"MAT file not found for {run_name}: {mat_path}")

    return build_metrics_csv_from_mat(mat_path, csv_out)


exports = []

for label, _start_time, _stop_time in SIM_RUN_PERIODS:
    best_run = f"{best_scenario}_{label}"
    worst_run = f"{worst_scenario}_{label}"

    best_csv = ensure_metrics_csv(best_run)
    worst_csv = ensure_metrics_csv(worst_run)

    df_best = pd.read_csv(best_csv)
    df_worst = pd.read_csv(worst_csv)

    sup_cols_best = sorted(
        [c for c in df_best.columns if c.endswith(" supply")],
        key=lambda x: int(re.search(r"Building (\d+)", x).group(1)),
    )
    ret_cols_best = sorted(
        [c for c in df_best.columns if c.endswith(" return")],
        key=lambda x: int(re.search(r"Building (\d+)", x).group(1)),
    )
    sup_cols_worst = sorted(
        [c for c in df_worst.columns if c.endswith(" supply")],
        key=lambda x: int(re.search(r"Building (\d+)", x).group(1)),
    )
    ret_cols_worst = sorted(
        [c for c in df_worst.columns if c.endswith(" return")],
        key=lambda x: int(re.search(r"Building (\d+)", x).group(1)),
    )

    sup_common = [c for c in sup_cols_best if c in sup_cols_worst]
    ret_common = [c for c in ret_cols_best if c in ret_cols_worst]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

    for col in sup_common:
        bid = re.search(r"Building (\d+)", col).group(1)
        axes[0].plot(
            df_best["hours_from_start"],
            df_best[col],
            lw=1.8,
            label=f"B{bid} supply (best)",
        )
        axes[0].plot(
            df_worst["hours_from_start"],
            df_worst[col],
            lw=1.8,
            ls="--",
            label=f"B{bid} supply (worst)",
        )

    axes[0].set_title(f"Supply Temperatures: Best vs Worst ({label})")
    axes[0].set_xlabel("Hours from start")
    axes[0].set_ylabel("Temperature (C)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=8, ncol=2)

    for col in ret_common:
        bid = re.search(r"Building (\d+)", col).group(1)
        axes[1].plot(
            df_best["hours_from_start"],
            df_best[col],
            lw=1.8,
            label=f"B{bid} return (best)",
        )
        axes[1].plot(
            df_worst["hours_from_start"],
            df_worst[col],
            lw=1.8,
            ls="--",
            label=f"B{bid} return (worst)",
        )

    axes[1].set_title(f"Return Temperatures: Best vs Worst ({label})")
    axes[1].set_xlabel("Hours from start")
    axes[1].set_ylabel("Temperature (C)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=8, ncol=2)

    fig.suptitle(f"5G Loop Temperature Comparison by Date: {label}", fontsize=14)
    fig.tight_layout()

    png_out = compare_dir / f"{label}_best_vs_worst_loop_temperature_compare.png"
    fig.savefig(png_out, dpi=220, bbox_inches="tight")
    plt.close(fig)

    # Quick scalar check: last building supply minimum for each case.
    last_bid = max(int(re.search(r"Building (\d+)", c).group(1)) for c in sup_common)
    last_col = f"Building {last_bid} supply"
    best_min = df_best[last_col].min()
    worst_min = df_worst[last_col].min()

    exports.append((label, png_out, last_col, best_min, worst_min))

print("Exported best-vs-worst date comparison plots:")
for label, png_out, last_col, best_min, worst_min in exports:
    delta = worst_min - best_min
    print(f"  {label}")
    print(f"    plot: {png_out}")
    print(
        f"    {last_col} min (C): best={best_min:.2f}, worst={worst_min:.2f}, diff={delta:+.2f}"
    )